# Module 2: Data I/O Operations

## Learning Objectives
By the end of this module, you will understand:
- Reading data from different file formats (CSV, JSON, Parquet, Delta)
- Writing data to different file formats
- Understanding write modes (append, overwrite, ignore, error)
- Configuring read/write options
- Working with compression
- Best practices for data I/O

---

## 1. File Format Overview

| Format | Use Case | Pros | Cons |
|--------|----------|------|------|
| **CSV** | Simple data exchange | Easy to read, widely used | No schema storage, slow for large files |
| **JSON** | API responses, nested data | Supports nested structures | Larger file size, slower |
| **Parquet** | Analytics, big data | Fast, compressed, schema stored | Less human-readable |
| **Delta Lake** | Data warehousing | ACID transactions, time travel | Databricks-specific |

---

## 2. Reading CSV Files

CSV (Comma-Separated Values) is one of the most common data formats.

In [ ]:
# Basic CSV read
df_csv = spark.read.csv("/path/to/file.csv", header=True, inferSchema=True)
df_csv.show()

### CSV Read Options:
- `header=True` - First row contains column names
- `inferSchema=True` - Automatically detect data types
- `delimiter=','` - Column separator (default is comma)
- `encoding='utf-8'` - Character encoding
- `nullValue='NA'` - Treat as null values

### Example: Handling different delimiters

In [ ]:
# Create sample CSV data
csv_data = """name,age,salary
Alice,28,70000
Bob,32,85000
Charlie,25,60000"""

# Write to DBFS
dbutils.fs.put("/dbfs/data/employees.csv", csv_data)

# Read CSV
df = spark.read.csv("/data/employees.csv", header=True, inferSchema=True)
df.show()
df.printSchema()

**Output:**
```
+-------+---+------+
|   name|age|salary|
+-------+---+------+
|  Alice| 28| 70000|
|    Bob| 32| 85000|
|Charlie| 25| 60000|
+-------+---+------+

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- salary: integer (nullable = true)
```

### Handling Data Quality Issues

In [ ]:
# CSV with missing values and header issues
messy_csv = """name|age|salary
Alice|28|70000
Bob||85000
Charlie|25|NA
Diana|35|95000"""

dbutils.fs.put("/dbfs/data/messy.csv", messy_csv)

# Read with proper options
df_messy = spark.read.csv(
    "/data/messy.csv",
    header=True,
    inferSchema=True,
    delimiter="|",           # Different delimiter
    nullValue="NA",          # Treat "NA" as null
    mode="PERMISSIVE"         # Allow bad rows (default)
)

df_messy.show()
df_messy.printSchema()

**Output:**
```
+-------+----+------+
|   name| age|salary|
+-------+----+------+
|  Alice|  28| 70000|
|    Bob|null| 85000|
|Charlie|  25|  null|
|  Diana|  35| 95000|
+-------+----+------+
```

---

## 3. Reading JSON Files

JSON (JavaScript Object Notation) is useful for nested/hierarchical data.

In [ ]:
# Create sample JSON data
json_data = """{ "name": "Alice", "age": 28, "salary": 70000 }
{ "name": "Bob", "age": 32, "salary": 85000 }
{ "name": "Charlie", "age": 25, "salary": 60000 }"""

dbutils.fs.put("/dbfs/data/employees.json", json_data)

# Read JSON - one JSON object per line
df_json = spark.read.json("/data/employees.json")
df_json.show()
df_json.printSchema()

**Output:**
```
+-------+---+------+
|    age|name|salary|
+-------+---+------+
|     28|Alice| 70000|
|     32|  Bob| 85000|
|     25|Charlie| 60000|
+-------+---+------+
```

**Note:** JSON automatically infers schema - column order might differ!

### Handling Nested JSON

In [ ]:
# JSON with nested structure
nested_json = """{ "name": "Alice", "contact": { "email": "alice@example.com", "phone": "555-1234" }, "salary": 70000 }
{ "name": "Bob", "contact": { "email": "bob@example.com", "phone": "555-5678" }, "salary": 85000 }"""

dbutils.fs.put("/dbfs/data/employees_nested.json", nested_json)

df_nested = spark.read.json("/data/employees_nested.json")
df_nested.show(truncate=False)  # Don't truncate long columns
df_nested.printSchema()

**Output:**
```
+-----+--------------------+------+
|name |contact             |salary|
+-----+--------------------+------+
|Alice|{alice@..., 555-1234}|70000|
|Bob  |{bob@..., 555-5678} |85000|
+-----+--------------------+------+

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- name: string (nullable = true)
 |-- salary: long (nullable = true)
```

---

## 4. Reading Parquet Files

Parquet is the **gold standard** for big data analytics - fast, compressed, and schema-aware.

In [ ]:
# First, create a DataFrame and save as Parquet
data = [("Alice", 28, 70000), ("Bob", 32, 85000), ("Charlie", 25, 60000)]
df_sample = spark.createDataFrame(data, ["name", "age", "salary"])

# Write to Parquet
df_sample.write.mode("overwrite").parquet("/data/employees.parquet")

# Read from Parquet
df_parquet = spark.read.parquet("/data/employees.parquet")
df_parquet.show()
df_parquet.printSchema()

**Output:**
```
+-------+---+------+
|   name|age|salary|
+-------+---+------+
|  Alice| 28| 70000|
|    Bob| 32| 85000|
|Charlie| 25| 60000|
+-------+---+------+

root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- salary: long (nullable = true)
```

### Why Parquet?
✅ **Columnar storage** - Only read needed columns (faster)
✅ **Compression** - Smaller file sizes
✅ **Schema storage** - No need for `inferSchema`
✅ **Type safety** - Data types preserved
✅ **Fast** - Optimized for analytics

---

## 5. Reading Delta Lake Files (Databricks)

Delta Lake is a storage layer built on Parquet that adds ACID transactions and more features.

In [ ]:
# Write to Delta Lake
df_sample = spark.createDataFrame(
    [("Alice", 28, 70000), ("Bob", 32, 85000), ("Charlie", 25, 60000)],
    ["name", "age", "salary"]
)

# Save as Delta
df_sample.write.mode("overwrite").format("delta").save("/data/employees_delta")

# Read Delta Lake
df_delta = spark.read.format("delta").load("/data/employees_delta")
df_delta.show()
df_delta.printSchema()

### Delta Lake Features:
✅ **ACID Transactions** - Multiple writes don't corrupt data
✅ **Schema Enforcement** - Prevents bad data
✅ **Time Travel** - Query historical versions
✅ **Unified Batch & Streaming** - Handle both in one format
✅ **Automatic Optimization** - Compaction, indexing

---

## 6. Writing Data - Write Modes

When writing data, you must specify what to do if data already exists.

In [ ]:
# Create sample data
df = spark.createDataFrame(
    [("Alice", 28), ("Bob", 32)],
    ["name", "age"]
)

# Mode 1: OVERWRITE (replace entire directory)
df.write.mode("overwrite").csv("/data/output/employees_v1.csv", header=True)
print("✓ Overwrite mode: File replaced if existed")

# Mode 2: APPEND (add to existing data)
df.write.mode("append").csv("/data/output/employees_v1.csv", header=True)
print("✓ Append mode: New rows added to existing file")

# Mode 3: IGNORE (do nothing if exists)
df.write.mode("ignore").csv("/data/output/employees_v1.csv", header=True)
print("✓ Ignore mode: File unchanged if exists")

# Mode 4: ERROR (raise error if exists) - default
# df.write.mode("error").csv("/data/output/employees_v1.csv", header=True)  # Would fail
print("✓ Error mode: Raises exception if file exists (default)")

| Mode | Behavior |
|------|----------|
| **overwrite** | Delete existing data and write new |
| **append** | Add new rows to existing data |
| **ignore** | Do nothing if path exists |
| **error** | Raise exception if path exists (default) |

---

## 7. Writing CSV Files

CSV write options:

In [ ]:
# Create sample data
df = spark.createDataFrame(
    [("Alice", 28, "Engineering"), ("Bob", 32, "Sales"), ("Charlie", 25, "Marketing")],
    ["name", "age", "department"]
)

# Write CSV with options
df.coalesce(1).write.mode("overwrite").csv(
    "/data/output/employees.csv",
    header=True,          # Include header
    delimiter=",",       # Column separator
    quote='"',           # Quote character for fields with special chars
    escape='\\',         # Escape character
    encoding="utf-8"     # Character encoding
)

# Read back and show
df_read = spark.read.csv("/data/output/employees.csv", header=True)
print("Written CSV content:")
df_read.show()

**Note:** `.coalesce(1)` - Writes to a single file instead of multiple partitions

---

## 8. Writing JSON Files

In [ ]:
# Create sample data
df = spark.createDataFrame(
    [("Alice", 28, 70000), ("Bob", 32, 85000)],
    ["name", "age", "salary"]
)

# Write JSON
df.coalesce(1).write.mode("overwrite").json("/data/output/employees.json")

# Read back and show
df_read = spark.read.json("/data/output/employees.json")
print("Written JSON content:")
df_read.show()

---

## 9. Writing Parquet Files (Recommended for Big Data)

In [ ]:
# Create sample data
df = spark.createDataFrame(
    [("Alice", 28, 70000), ("Bob", 32, 85000), ("Charlie", 25, 60000)],
    ["name", "age", "salary"]
)

# Write Parquet with compression
df.write.mode("overwrite").parquet(
    "/data/output/employees.parquet",
    compression="snappy"  # Or: gzip, lz4, uncompressed, brotli
)

# Read back and verify
df_read = spark.read.parquet("/data/output/employees.parquet")
print("Parquet file written successfully!")
print("File size: (Parquet files are often 10-100x smaller than CSV)")
df_read.show()
df_read.printSchema()

### Compression Algorithms:
- **snappy** (default) - Fast, moderate compression
- **gzip** - Slower, better compression
- **lz4** - Fastest, minimal compression
- **brotli** - Excellent compression, slower
- **uncompressed** - No compression (fastest write)

---

## 10. Writing Delta Lake Files

In [ ]:
# Create sample data
df = spark.createDataFrame(
    [("Alice", 28, "Engineering"), ("Bob", 32, "Sales")],
    ["name", "age", "department"]
)

# Write as Delta with overwrite
df.write.mode("overwrite").format("delta").save("/data/output/employees_delta")

# Append more data
df_append = spark.createDataFrame(
    [("Charlie", 25, "Marketing")],
    ["name", "age", "department"]
)
df_append.write.mode("append").format("delta").save("/data/output/employees_delta")

# Read all data
df_result = spark.read.format("delta").load("/data/output/employees_delta")
print("Delta Lake file written and appended successfully!")
df_result.show()

---

## 11. Complete Read/Write Workflow

Let's see a realistic end-to-end example:

In [ ]:
# Step 1: Create sample CSV data
csv_data = """product_id,product_name,price,quantity
1,Laptop,1200,5
2,Phone,800,15
3,Tablet,400,20
4,Monitor,300,8"""

dbutils.fs.put("/dbfs/data/products.csv", csv_data)

# Step 2: Read CSV
df = spark.read.csv("/data/products.csv", header=True, inferSchema=True)
print("Step 1: Read CSV")
df.show()

# Step 3: Add a calculated column (inventory value)
from pyspark.sql.functions import col
df_enriched = df.withColumn("inventory_value", col("price") * col("quantity"))
print("\nStep 2: Add calculated column")
df_enriched.show()

# Step 4: Write to multiple formats
# Write as CSV
df_enriched.coalesce(1).write.mode("overwrite").csv("/data/output/products_enriched.csv", header=True)
# Write as Parquet
df_enriched.write.mode("overwrite").parquet("/data/output/products_enriched.parquet")
# Write as Delta
df_enriched.write.mode("overwrite").format("delta").save("/data/output/products_enriched_delta")

print("\nStep 3: Written to CSV, Parquet, and Delta formats")

# Step 5: Read back from different formats to verify
df_csv_verify = spark.read.csv("/data/output/products_enriched.csv", header=True, inferSchema=True)
df_parquet_verify = spark.read.parquet("/data/output/products_enriched.parquet")
df_delta_verify = spark.read.format("delta").load("/data/output/products_enriched_delta")

print("\nStep 4: Verified - All formats contain correct data")
print(f"CSV rows: {df_csv_verify.count()}")
print(f"Parquet rows: {df_parquet_verify.count()}")
print(f"Delta rows: {df_delta_verify.count()}")

**Output:**
```
Step 1: Read CSV
+----------+----------+-----+--------+
|product_id|product_name|price|quantity|
+----------+----------+-----+--------+
|         1|    Laptop|1200|       5|
|         2|     Phone| 800|      15|
|         3|     Tablet| 400|      20|
|         4|    Monitor| 300|       8|
+----------+----------+-----+--------+

Step 2: Add calculated column
+----------+-----------+-----+--------+----------------+
|product_id|product_name|price|quantity|inventory_value|
+----------+-----------+-----+--------+----------------+
|         1|     Laptop|1200|       5|            6000|
|         2|      Phone| 800|      15|           12000|
|         3|      Tablet| 400|      20|            8000|
|         4|     Monitor| 300|       8|            2400|
+----------+-----------+-----+--------+----------------+

Step 3: Written to CSV, Parquet, and Delta formats

Step 4: Verified - All formats contain correct data
CSV rows: 4
Parquet rows: 4
Delta rows: 4
```

---

## 12. Quick Reference - Read/Write Syntax

### Reading Data
```python
# CSV
df = spark.read.csv("path", header=True, inferSchema=True)

# JSON
df = spark.read.json("path")

# Parquet
df = spark.read.parquet("path")

# Delta
df = spark.read.format("delta").load("path")
```

### Writing Data
```python
# CSV
df.write.mode("overwrite").csv("path", header=True)

# JSON
df.write.mode("overwrite").json("path")

# Parquet
df.write.mode("overwrite").parquet("path")

# Delta
df.write.mode("overwrite").format("delta").save("path")
```

---

## 13. Mini Challenges

### Challenge 1: Create and Read CSV
1. Create a CSV file with 5 products (id, name, price, category)
2. Read it into a DataFrame
3. Print the schema
4. Show all rows

### Challenge 2: Format Conversion
1. Create a DataFrame with some data
2. Write it as CSV
3. Write the same DataFrame as Parquet
4. Write the same DataFrame as Delta
5. Read from each format and verify they have same number of rows

### Challenge 3: Write Modes
1. Create and write a DataFrame with mode "overwrite"
2. Create a new DataFrame with different data
3. Write it with mode "append" to the same location
4. Read the result and count rows (should see both old and new data)
5. Try mode "ignore" - what happens?

---

## Key Takeaways

✅ **CSV** - Easy but slow for big data

✅ **JSON** - Good for nested/API data

✅ **Parquet** - Best for analytics (fast, compressed, schema stored)

✅ **Delta** - Production choice (ACID, time travel, reliability)

✅ **Write Modes** - Choose right mode (overwrite/append/ignore/error)

✅ **Always use `.coalesce(1)`** when writing CSV for single file output

✅ **Schema inference** is convenient but can be slow - use explicit schema for large files

---

## Next Steps
In Module 3, we'll explore **how to view and explore DataFrames** - selecting columns, renaming, and understanding your data!